# FastWoe Softmax WOE

Author: https://www.github.com/xRiskLab

`FastWoe(conditional=True)` measures each feature's weight inside the cell picked out by the features before it, by counting. With a few features that works; with more, the cells run thin and fall back to marginal weights.

**`SoftmaxWoe` is a generative classifier: per class, an autoregressive chain of penalized multinomial logistic regressions, fitted by maximum likelihood of the features given the class; its per-feature weights are Good's conditional weights of evidence under that model and sum exactly to the posterior log-odds.**

`SoftmaxWoe` replaces the counts with a small model per feature. Per class, each applicant's profile is a leaf in a tree with one level per feature (the hierarchical softmax of Goodfellow et al., 2016, §12.4.3.2). Each node is a softmax that predicts the bin of the next feature from the earlier bins, fitted once among bads and once among goods. The weight at a node is

$$W(\text{bad} : \text{bin}_i \mid \text{bin}_{<i}) = \log P(\text{bin}_i \mid \text{bin}_{<i}, \text{bad}) - \log P(\text{bin}_i \mid \text{bin}_{<i}, \text{good})$$

and the weights add up exactly to the profile's log-likelihood ratio.

The data here is synthetic, from a known generator, so every weight has a true value to compare against.

**Set `C` in the next cell, then run all.** Small `C` shrinks every node toward the plain bin shares (the marginal scorecard). Large `C` lets each node follow the earlier bins freely.

In [1]:
C = 1.0  # inverse penalty strength: smaller = more shrinkage
N_TRAIN = 5000
SEED = 7

## The generator

Three binned features in a fixed order: delinquency, then utilization, then inquiries. In each class the bin of every feature depends on the earlier bins, and that dependence differs between bads and goods. This is what makes marginal WOE double count.

In [2]:
import itertools
import warnings

import numpy as np
import pandas as pd

from fastwoe import FastWoe, SoftmaxWoe

pd.set_option("display.width", 60)

LEVELS = {
    "delinq": ["no", "yes"],
    "util": ["<30", "30-80", ">80"],
    "inq": ["0", "1-2", "3+"],
}
ORDER = list(LEVELS)
PRIOR_BAD = 0.10

rng = np.random.default_rng(SEED)
# true node scores: intercept per bin and class, plus one effect per earlier bin
TRUE = {}
for i, f in enumerate(ORDER):
    K = len(LEVELS[f])
    n_in = sum(len(LEVELS[g]) for g in ORDER[:i])
    for h in (1, 0):
        a = rng.normal(0, 0.8, K) + (np.linspace(-0.6, 0.6, K) if h else 0)
        B = rng.normal(0, 0.9, (n_in, K)) * (1.6 if h else 1.0)
        TRUE[(i, h)] = (a, B)


def softmax(z: np.ndarray) -> np.ndarray:
    """Softmax of each row."""
    z = z - z.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)


def onehot(df: pd.DataFrame, feats: list[str]) -> np.ndarray:
    """One-hot of the given features, every bin kept, in LEVELS order."""
    blocks = [np.eye(len(LEVELS[f]))[pd.Categorical(df[f], LEVELS[f]).codes] for f in feats]
    return np.hstack(blocks) if blocks else np.zeros((len(df), 0))


def true_probs(i: int, h: int, df: pd.DataFrame) -> np.ndarray:
    """True P(bin of feature i | earlier bins, class h): rows x bins."""
    a, B = TRUE[(i, h)]
    return softmax(a + onehot(df, ORDER[:i]) @ B)


def true_weights(df: pd.DataFrame) -> np.ndarray:
    """True weight of every feature for every row."""
    W = np.zeros((len(df), len(ORDER)))
    rows = np.arange(len(df))
    for i, f in enumerate(ORDER):
        k = pd.Categorical(df[f], LEVELS[f]).codes
        W[:, i] = np.log(true_probs(i, 1, df)[rows, k] / true_probs(i, 0, df)[rows, k])
    return W


def sample(n: int) -> pd.DataFrame:
    """Draw n applicants from the generator."""
    y = (rng.random(n) < PRIOR_BAD).astype(int)
    df = pd.DataFrame({"y": y})
    for i, f in enumerate(ORDER):
        df[f] = ""
        for h in (1, 0):
            m = (df.y == h).to_numpy()
            p = true_probs(i, h, df[m])
            k = (p.cumsum(1) > rng.random((m.sum(), 1))).argmax(1)
            df.loc[m, f] = np.array(LEVELS[f])[k]
    return df[ORDER + ["y"]]


train, test = sample(N_TRAIN), sample(20_000)
print(f"train {len(train)}: {train.y.sum()} bad, {(1 - train.y).sum()} good")
print(f"{np.prod([len(v) for v in LEVELS.values()])} possible profiles")
train.head()

train 5000: 501 bad, 4499 good
18 possible profiles


,delinq,util,inq,y
0,no,30-80,1-2,0
1,no,30-80,1-2,0
2,no,>80,0,0
3,no,30-80,1-2,0
4,no,>80,1-2,0


## Fit SoftmaxWoe

One node model per feature and class. The first node has no earlier bins, so it is the bin shares (with a 0.5 pseudo-count). Earlier bins enter one-hot with every bin kept, so the penalty treats all bins alike. `order` fixes the conditioning order; unlike counted conditional WOE, it changes the score as well as the attribution.

In [3]:
model = SoftmaxWoe(order=ORDER, C=C).fit(train[ORDER], train.y)
prior = model.prior_log_odds_
print(f"C = {C}, prior log-odds {prior:+.3f}")
model.transform(test[ORDER].head())

C = 1.0, prior log-odds -2.195


,delinq,util,inq
0,-0.974851,1.629423,-3.066557
1,0.679855,-0.308553,0.851306
2,-0.974851,-1.875120,-3.840206
3,-0.974851,-1.875120,-3.840206
4,-0.974851,-1.875120,-3.840206


## The bins at each node

For every path of earlier bins: the fitted P(bin | bad) and P(bin | good) from `node_proba`, the weight W, the true weight, and how many training bads and goods sit on that path. Re-run with another `C` to watch the weights move.

In [4]:
def show_node(f: str) -> None:
    """Print the fitted and true weights of feature f for every path of earlier bins."""
    i = ORDER.index(f)
    print(f"=== node {i}: {f} ===")
    for path in itertools.product(*[LEVELS[g] for g in ORDER[:i]]):
        given = dict(zip(ORDER[:i], path))
        rows = pd.DataFrame([given] * len(LEVELS[f]), columns=ORDER[:i])
        rows[f] = LEVELS[f]
        p = model.node_proba(rows, f)
        k = np.arange(len(LEVELS[f]))
        t1, t0 = true_probs(i, 1, rows)[k, k], true_probs(i, 0, rows)[k, k]
        on = (train[ORDER[:i]] == pd.Series(given)).all(1) if i else train.y >= 0
        nb, ng = int(train.y[on].sum()), int((1 - train.y[on]).sum())
        label = ", ".join(f"{g}={v}" for g, v in given.items()) or "(nothing yet)"
        print(f"given {label}   [{nb} bad, {ng} good]")
        out = pd.DataFrame(
            {
                "P|bad": p.p_event.to_numpy(),
                "P|good": p.p_nonevent.to_numpy(),
                "W": np.log(p.p_event / p.p_nonevent).to_numpy(),
                "true W": np.log(t1 / t0),
            },
            index=LEVELS[f],
        )
        print(out.round(3).to_string(), "\n")


for f in ORDER:
    show_node(f)

=== node 0: delinq ===
given (nothing yet)   [501 bad, 4499 good]
     P|bad  P|good      W  true W
no    0.23    0.61 -0.975  -1.174
yes   0.77    0.39  0.680   0.757 

=== node 1: util ===
given delinq=no   [115 bad, 2744 good]
       P|bad  P|good      W  true W
<30    0.647   0.127  1.629   1.729
30-80  0.089   0.581 -1.875  -2.258
>80    0.264   0.292 -0.101  -0.282 

given delinq=yes   [386 bad, 1755 good]
       P|bad  P|good      W  true W
<30    0.227   0.068  1.199   0.881
30-80  0.170   0.111  0.429   0.636
>80    0.603   0.821 -0.309  -0.310 

=== node 2: inq ===
given delinq=no, util=<30   [75 bad, 348 good]
     P|bad  P|good      W  true W
0    0.207   0.527 -0.937  -0.910
1-2  0.022   0.468 -3.067  -3.158
3+   0.772   0.005  5.119   5.855 

given delinq=no, util=30-80   [10 bad, 1596 good]
     P|bad  P|good      W  true W
0    0.469   0.182  0.949   0.932
1-2  0.017   0.808 -3.840  -4.852
3+   0.514   0.010  3.906   3.957 

given delinq=no, util=>80   [30 bad, 800 good

## One applicant, step by step

Change `who` to follow another applicant from the test set. `transform` gives one weight per feature; the running total starts at the prior log-odds.

In [5]:
who: int = 0
app = test.iloc[[who]]
W = model.transform(app[ORDER]).to_numpy()[0]
Wt = true_weights(app)[0]
print(
    ", ".join(f"{f}={app[f].iloc[0]}" for f in ORDER),
    "| bad" if app.y.iloc[0] else "| good",
)
print(f"{'step':8}{'bin':>7}{'W':>8}{'true':>8}{'running':>9}")
run = prior
for i, f in enumerate(ORDER):
    run += W[i]
    print(f"{f:8}{app[f].iloc[0]:>7}{W[i]:+8.3f}{Wt[i]:+8.3f}{run:+9.3f}")
print(f"P(bad) = {1 / (1 + np.exp(-run)):.3f}")
print(f"predict_proba: {model.predict_proba(app[ORDER])[0, 1]:.3f}")

delinq=no, util=<30, inq=1-2 | good
step        bin       W    true  running
delinq       no  -0.975  -1.174   -3.170
util        <30  +1.629  +1.729   -1.540
inq         1-2  -3.067  -3.158   -4.607
P(bad) = 0.010
predict_proba: 0.010


## Inside one node: the softmax step by step

What a single node does for a single applicant. Take someone who is delinquent with utilization over 80%: which inquiries bin comes next? The node answers twice, with the bads' coefficients and with the goods', in four steps:

1. **Score** each bin: intercept + the effect of each earlier bin. Scores can be any number.
2. **Exponentiate**: every score becomes positive, and higher scores grow faster.
3. **Divide by the total**: the three numbers now sum to 1, a probability for each bin. Any such triple is one point in the triangle of all possible distributions over three bins (the *simplex*): corners are certainty, the center is ⅓ each. Distances on the triangle are not evidence: near an edge a small gap can be a large log ratio, so the evidence is always read as a log ratio of coordinates.
4. **Weight**: for the bin the applicant is actually in, W = log P(bin | bad) − log P(bin | good).

The node models are scikit-learn `LogisticRegression` objects in `model.nodes_`, keyed by (feature, class). Their inputs are the earlier features one-hot in `model.levels_` order.

In [6]:
path = {"delinq": "yes", "util": ">80"}  # the earlier bins
observed = "3+"  # the inquiries bin the applicant has

f = "inq"
i = ORDER.index(f)
cols = [(g, b) for g in ORDER[:i] for b in model.levels_[g]]
x = np.array([1.0 if path[g] == b else 0.0 for g, b in cols])
active = [(g, b) for g, b in cols if path[g] == b]
bins = model.levels_[f]
P = {}
for h, name in ((1, "bad"), (0, "good")):
    m = model.nodes_[(f, h)]
    z = m.intercept_ + m.coef_ @ x
    e = np.exp(z)
    P[h] = e / e.sum()
    print(f"--- if the applicant is {name} (C = {C}) ---")
    print("1. score = intercept + " + " + ".join(f"{g}={b}" for g, b in active))
    for k, b in enumerate(bins):
        parts = [m.intercept_[k]] + [m.coef_[k, cols.index(a)] for a in active]
        print(f"   {b:>3}: " + " ".join(f"{v:+.2f}" for v in parts) + f" = {z[k]:+.2f}")
    print("2. exponentiate")
    for k, b in enumerate(bins):
        print(f"   {b:>3}: exp({z[k]:+.2f}) = {e[k]:.2f}")
    print(f"3. divide by the total {e.sum():.2f}")
    for k, b in enumerate(bins):
        print(f"   {b:>3}: {e[k]:.2f} / {e.sum():.2f} = {P[h][k]:.3f}")
    print()

k = bins.index(observed)
print(
    f"4. W(bad : inq={observed} | {path}) = "
    f"log({P[1][k]:.4f} / {P[0][k]:.4f}) = {np.log(P[1][k] / P[0][k]):+.2f}"
)
row = pd.DataFrame([{**path, f: observed}])
print(f"   model.transform gives the same:   {model.transform(row)[f].iloc[0]:+.2f}")
marginal_w = FastWoe().fit(train[ORDER], train.y).transform(row)[f].iloc[0]
print(f"   measured against the whole book:  {marginal_w:+.2f}")

--- if the applicant is bad (C = 1.0) ---
1. score = intercept + delinq=yes + util=>80
     0: +0.48 -0.22 +0.03 = +0.29
   1-2: -0.79 +1.33 -0.07 = +0.47
    3+: +0.31 -1.11 +0.04 = -0.76
2. exponentiate
     0: exp(+0.29) = 1.33
   1-2: exp(+0.47) = 1.60
    3+: exp(-0.76) = 0.47
3. divide by the total 3.40
     0: 1.33 / 3.40 = 0.392
   1-2: 1.60 / 3.40 = 0.470
    3+: 0.47 / 3.40 = 0.138

--- if the applicant is good (C = 1.0) ---
1. score = intercept + delinq=yes + util=>80
     0: +1.20 +0.28 -0.23 = +1.25
   1-2: +0.93 -0.58 -0.40 = -0.05
    3+: -2.13 +0.30 +0.63 = -1.20
2. exponentiate
     0: exp(+1.25) = 3.48
   1-2: exp(-0.05) = 0.95
    3+: exp(-1.20) = 0.30
3. divide by the total 4.74
     0: 3.48 / 4.74 = 0.736
   1-2: 0.95 / 4.74 = 0.201
    3+: 0.30 / 4.74 = 0.064

4. W(bad : inq=3+ | {'delinq': 'yes', 'util': '>80'}) = log(0.1377 / 0.0637) = +0.77
   model.transform gives the same:   +0.77


   measured against the whole book:  +2.14


## Why the weights add up exactly

Each node is a proper distribution over its bins, so in each class the product along the path is a proper distribution over all profiles. Both sums below are 1, so prior + ΣW is the log posterior odds under the fitted chain.

In [7]:
grid = pd.DataFrame(list(itertools.product(*LEVELS.values())), columns=ORDER)
lik = {"p_event": np.ones(len(grid)), "p_nonevent": np.ones(len(grid))}
for f in ORDER:
    p = model.node_proba(grid, f)
    for col in lik:
        lik[col] *= p[col].to_numpy()
for col, name in (("p_event", "bad"), ("p_nonevent", "good")):
    print(f"sum of P(profile | {name}) over {len(grid)} profiles = {lik[col].sum():.6f}")

sum of P(profile | bad) over 18 profiles = 1.000000
sum of P(profile | good) over 18 profiles = 1.000000


## How C moves things

Holdout log loss and the average error of the weights against the truth, across a range of `C`. Marginal WOE (every node cut back to its bin shares) and the true weights are the two ends. Counted conditional WOE (`FastWoe(conditional=True)`, threshold 1) is shown for comparison. Its log loss is close to the softmax chain's, but its weights are noisier: each one is read off a single cell, some cells hold only a handful of bads, and two hold none at all, so they fall back to the marginal weight. The softmax chain borrows strength across cells through the shared node model.

In [8]:
def log_loss(y: np.ndarray, s: np.ndarray) -> float:
    """Binary log loss from log-odds scores."""
    p = np.clip(1 / (1 + np.exp(-s)), 1e-12, 1 - 1e-12)
    return float(-np.mean(y * np.log(p) + (1 - y) * np.log(1 - p)))


y, Wt = test.y.to_numpy(), true_weights(test)
X_train, X_test = train[ORDER], test[ORDER]


def row(name: str, W: np.ndarray, base: float = prior) -> dict:
    """Holdout log loss and mean absolute error of the weights against the truth."""
    return {
        "model": name,
        "log loss": log_loss(y, base + W.sum(1)),
        "W error": np.abs(W - Wt).mean(),
    }


rows = [row("marginal WOE", FastWoe().fit(X_train, train.y).transform(X_test).to_numpy())]
conditional = FastWoe(conditional=True, conditional_order=ORDER, conditional_min_count=1)
with warnings.catch_warnings():  # two empty cells fall back; reported below
    warnings.simplefilter("ignore")
    conditional.fit(X_train, train.y)
rows.append(row("conditional WOE", conditional.transform(X_test).to_numpy()))
print(f"conditional WOE: {len(conditional.conditional_fallbacks_)} cells fell back to marginal")
for c in (0.001, 0.01, 0.1, 1, 10, 100):
    W = SoftmaxWoe(order=ORDER, C=c).fit(X_train, train.y).transform(X_test).to_numpy()
    rows.append(row(f"softmax C={c:g}", W))
rows.append(row("true weights", Wt, np.log(PRIOR_BAD / (1 - PRIOR_BAD))))
pd.DataFrame(rows).set_index("model").round(4)

conditional WOE: 2 cells fell back to marginal


,log loss,W error
model,,
marginal WOE,0.2569,1.0021
conditional WOE,0.2057,0.6724
softmax C=0.001,0.2370,0.8823
softmax C=0.01,0.2158,0.7364
softmax C=0.1,0.2046,0.4728
softmax C=1,0.2022,0.2484
softmax C=10,0.2020,0.1760
softmax C=100,0.2020,0.1684
true weights,0.2001,0.0000
